In [1]:
# In this dataset our desired target for classification task will be converted variable - has the client signed up to the platform or not.

In [55]:
import pandas as pd
import matplotlib as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

%matplotlib inline

In [41]:
# The dataset
!wget -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

File ‘course_lead_scoring_2026.csv’ already there; not retrieving.



In [42]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [43]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [44]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [45]:
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
numerical_columns = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical_columns

['lead_source', 'industry', 'employment_status', 'location']

In [46]:
df[categorical_columns] = df[categorical_columns].fillna('NA')
display(df.head())
df.isnull().sum()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


lead_source                   0
industry                      0
employment_status             0
location                      0
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [47]:
df[numerical_columns] = df[numerical_columns].fillna(0.0)
display(df.head())
df.isnull().sum()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1

What is the most frequent observation (mode) for the column `industry`?
- `NA`
- `technology`
- `healthcare`
- `retail`

In [48]:
df.industry.mode()[0]

'technology'

Answer: `technology`

## Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?
- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

In [49]:
df[numerical_columns].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


Answer: `interaction_count` and `lead_score`

## Split the data

Split the data with these exact calls:
```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```
Make sure that the target value `converted` is not in your dataframe.

In [50]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Question 3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?
- `industry`
- `location`
- `lead_source`
- `employment_status`

In [51]:
for c in categorical_columns:
    print(c, round(mutual_info_score(df_train[c], y_train), 2))
# Notice that sklearn's function mutual_info_score can take in a pandas series and a numpy array and still work!
# Great abstraction work from the scikit-learn maintainers.

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


Answer: `lead_source`

## Question 4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?
- 0.55
- 0.65
- 0.75
- 0.85

In [53]:
all_features = categorical_columns + numerical_columns

def train(features):
    dv = DictVectorizer(sparse=False)
    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.fit_transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    accuracy = accuracy_score(y_val, model.predict(X_val))

    return accuracy


In [59]:
accuracy = train(all_features)
rounded_accuracy = round(accuracy, 2)
print(accuracy, rounded_accuracy)

0.645 0.65


Answer: 0.65

## Question 5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following features has the smallest difference?
- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

Note: The difference doesn't have to be positive.

In [65]:
differences = {}

for f in all_features:
    accuracy_without_f = train([i for i in all_features if i != f])
    difference = accuracy - accuracy_without_f
    differences[f] = difference
    print(f'{f:30s} accuracy without this feature = {accuracy_without_f}     difference = {difference}')

lead_source                    accuracy without this feature = 0.642     difference = 0.0030000000000000027
industry                       accuracy without this feature = 0.645     difference = 0.0
employment_status              accuracy without this feature = 0.645     difference = 0.0
location                       accuracy without this feature = 0.645     difference = 0.0
annual_income                  accuracy without this feature = 0.724     difference = -0.07899999999999996
number_of_courses_viewed       accuracy without this feature = 0.643     difference = 0.0020000000000000018
interaction_count              accuracy without this feature = 0.601     difference = 0.04400000000000004
lead_score                     accuracy without this feature = 0.644     difference = 0.0010000000000000009


In [75]:
sorted(
    (item for item in differences.items() if item[0] in {'lead_source', 'number_of_courses_viewed', 'interaction_count'}),
    key=lambda item: item[1]
)

[('number_of_courses_viewed', 0.0020000000000000018),
 ('lead_source', 0.0030000000000000027),
 ('interaction_count', 0.04400000000000004)]

Answer: `'number_of_courses_viewed'`

## Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?
- 0.000001
- 0.00001
- 0.0001
- 0.001

Note: If there are multiple options, select the smallest C.

In [78]:
c_candidate_values = [0.000001, 0.00001, 0.0001, 0.001]

def train(features, c):
    dv = DictVectorizer(sparse=False)
    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.fit_transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    accuracy = accuracy_score(y_val, model.predict(X_val))

    return accuracy

for c_value in c_candidate_values:
    print(c_value, round(train(all_features, c_value), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


Answer: 0.001

<style>
/* 1. Forces wide tables and standard HTML outputs to fit */
.output_subarea, .output_html, table.dataframe {
    max-width: 100% !important;
    overflow-x: auto !important;
    display: block !important;
}

/* 2. Forces standard stdout code prints, stderr outputs (like wget), 
      and Python Traceback errors to wrap text safely */
div.output_area pre, 
div.output_subarea.output_stream.output_stderr pre, 
div.output_subarea.output_error pre {
    white-space: pre-wrap !important;
    word-break: break-word !important;
}
</style>